# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abc085455-byte/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [4]:
# Full fact table schema (no "..." truncation)
schema_fact = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet') LIMIT 1
""").df()
print(schema_fact.to_string())

# dim_content — correct path this time
schema_content = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/dim_content.parquet') LIMIT 1").df()
print(schema_content.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fields = {
    "feature": ["gsc_clicks_trailing_28d", "gsc_impressions_trailing_28d",
                "gsc_avg_position_trailing_28d", "content_age_days", "click_trend_28v28"],
    "label": ["needs_refresh"],
    "context": ["client_hash_id", "client_has_gsc", "content_type", "last_optimized_date"],
    "excluded": ["gsc_data_available = FALSE (missing, not a real zero)",
                 "is_deleted = TRUE or is_published = FALSE (not actionable)",
                 "content_age_days < 90 (early-ramp pages)"]
}
for bucket, items in fields.items():
    print(f"{bucket.upper()}: {items}")

FEATURE: ['gsc_clicks_trailing_28d', 'gsc_impressions_trailing_28d', 'gsc_avg_position_trailing_28d', 'content_age_days', 'click_trend_28v28']
LABEL: ['needs_refresh']
CONTEXT: ['client_hash_id', 'client_has_gsc', 'content_type', 'last_optimized_date']
EXCLUDED: ['gsc_data_available = FALSE (missing, not a real zero)', 'is_deleted = TRUE or is_published = FALSE (not actionable)', 'content_age_days < 90 (early-ramp pages)']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
FACT_MARCH = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"
FACT_FEB   = f"{REL}/fact_content_daily_performance/month=2026-02/data_0.parquet"
FACT_APRIL = f"{REL}/fact_content_daily_performance/month=2026-04/data_0.parquet"

# (a) GRAIN
q1 = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS n_rows
    FROM read_parquet('{FACT_MARCH}')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()
print("Duplicate (content_hash_id, client_hash_id, report_date) rows:", len(q1))

# (b) ROW COUNT + DATE SPAN
q2 = con.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM read_parquet('{FACT_MARCH}')
""").df()
display(q2)

# (c) AVAILABILITY
q3 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
    FROM read_parquet('{FACT_MARCH}')
""").df()
q3["pct_available"] = 100 * q3.available_rows / q3.total_rows
display(q3)

# ---- DEDUPE dim_content to content_hash_id grain first ----
content_attrs = con.sql(f"""
    SELECT content_hash_id,
           MIN(content_created_date) AS content_created_date,
           ANY_VALUE(content_type) AS content_type,
           MAX(last_optimized_date) AS last_optimized_date,
           BOOL_OR(is_deleted) AS is_deleted,
           BOOL_AND(is_published) AS is_published
    FROM read_parquet('{REL}/dim_content.parquet')
    GROUP BY content_hash_id
""").df()

# ---- 5-FEATURE FRAME ----
feat = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_clicks) AS gsc_clicks_trailing_28d,
           SUM(gsc_impressions) AS gsc_impressions_trailing_28d,
           AVG(gsc_avg_position) AS gsc_avg_position_trailing_28d
    FROM read_parquet('{FACT_MARCH}')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-28'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

feat = feat.merge(content_attrs, on="content_hash_id", how="left")
feat["content_age_days"] = (pd.Timestamp("2026-03-28") - pd.to_datetime(feat["content_created_date"])).dt.days

prior = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS clicks_prior_28d
    FROM read_parquet('{FACT_FEB}')
    WHERE report_date BETWEEN DATE '2026-01-29' AND DATE '2026-02-25'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
feat = feat.merge(prior, on=["content_hash_id","client_hash_id"], how="left")
feat["click_trend_28v28"] = feat.gsc_clicks_trailing_28d / feat.clicks_prior_28d.replace(0, pd.NA)

# apply exclusions from section 2
feat = feat[(feat.content_age_days >= 90) &
            (feat.is_deleted != True) &
            (feat.is_published != False)]
feat.head()

# ---- THE TRAP ----
future = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS future_clicks_28d
    FROM read_parquet('{FACT_APRIL}')
    WHERE report_date BETWEEN DATE '2026-03-29' AND DATE '2026-04-25'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
feat = feat.merge(future, on=["content_hash_id","client_hash_id"], how="left")
feat["needs_refresh"] = (feat.future_clicks_28d < 0.8 * feat.gsc_clicks_trailing_28d).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest = ["gsc_clicks_trailing_28d","gsc_impressions_trailing_28d",
          "gsc_avg_position_trailing_28d","content_age_days","click_trend_28v28"]
d = feat.dropna(subset=honest+["needs_refresh"])
Xtr,Xte,ytr,yte = train_test_split(d[honest], d.needs_refresh, test_size=0.3, random_state=0)
m = LogisticRegression(max_iter=1000).fit(Xtr,ytr)
print("HONEST AUC:", roc_auc_score(yte, m.predict_proba(Xte)[:,1]))

leaky = honest + ["future_clicks_28d"]
d2 = feat.dropna(subset=leaky+["needs_refresh"])
Xtr2,Xte2,ytr2,yte2 = train_test_split(d2[leaky], d2.needs_refresh, test_size=0.3, random_state=0)
m2 = LogisticRegression(max_iter=1000).fit(Xtr2,ytr2)
print("LEAKY AUC (jumps toward 1.0):", roc_auc_score(yte2, m2.predict_proba(Xte2)[:,1]))

leaky.remove("future_clicks_28d")
print("Deleted the leak. Honest feature set kept:", honest)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (content_hash_id, client_hash_id, report_date) rows: 0


,n_rows,n_content_items,n_clients,first_date,last_date
0,9841378,331437,55,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,pct_available
0,9841378,3611061,36.692636


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HONEST AUC: 0.7526970566191836
LEAKY AUC (jumps toward 1.0): 0.9999803525195231
Deleted the leak. Honest feature set kept: ['gsc_clicks_trailing_28d', 'gsc_impressions_trailing_28d', 'gsc_avg_position_trailing_28d', 'content_age_days', 'click_trend_28v28']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
panel_check = con.sql(f"""
    SELECT access_profile, COUNT(*) AS n_clients,
           SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS n_missing_gsc_start
    FROM read_parquet('{REL}/dim_clients.parquet')
    GROUP BY access_profile
    ORDER BY n_clients DESC
""").df()
panel_check

,access_profile,n_clients,n_missing_gsc_start
0,gsc_and_ga4,53,3.0
1,no_search_or_analytics_access,26,23.0
2,gsc_only,14,4.0
3,source_only_missing_client_dimension,10,6.0
4,ga4_only,1,1.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.